In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"D:\DS115-VIS\.venv\Project1_Predictive-Maintenance\ai4i_fused.csv")
df.head()


In [ ]:
# Clean trailing and leading spaces from column names
df.columns = df.columns.str.strip()

In [ ]:
# 2. Sort data for time-series compatibility
# Chronological order is mandatory for accurate rolling calculations
df = df.sort_values(by=['date', 'UDI']).reset_index(drop=True)

In [ ]:
# 3. Select Target Telemetry Features
# These are the main continuous sensor and weather columns from image
sensor_features = [
    'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'aavg_wind_speed_kmh', 'avg_sea_level_pres_hpa'
]

In [ ]:
# Define the rolling window size (Operational window size - e.g., 5 logs)
window_size = 5
df.head(5)

In [ ]:
print("Starting feature engineering...")


In [ ]:
# 4. Engineer Rolling Mean, Standard Deviation, and Variance
# Perform calculations inside a new DataFrame
engineered_features = pd.DataFrame(index=df.index)


In [ ]:
for col in sensor_features:
    # Check if the column exists in the dataset
    if col in df.columns:
        # Rolling Mean
        engineered_features[f'{col}_roll_mean'] = df[col].rolling(window=window_size).mean()
        
        # Rolling Standard Deviation
        engineered_features[f'{col}_roll_std'] = df[col].rolling(window=window_size).std()
        
        # Rolling Variance
        engineered_features[f'{col}_roll_var'] = df[col].rolling(window=window_size).var()

In [ ]:
# 5. Combine with original metadata and targets
# Includes UDI, Product ID, Machine failure, and specific failure modes
meta_cols = ['UDI', 'Product ID', 'Type', 'date', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
# Filter out names to match what is present in the dataset
meta_cols = [c for c in meta_cols if c in df.columns]

final_df = pd.concat([df[meta_cols], engineered_features], axis=1)


In [ ]:
# 6. Drop NaN rows generated by the rolling window warmup buffer
final_df.dropna(inplace=True)
df.head(5)


In [ ]:
# 7. Verify the output structure and save the file
print(f"Processed dataset shape: {final_df.shape}")
print(final_df.head())

# Save the final engineered dataset
final_df.to_csv("engineered_iot_weather_dataset.csv", index=False)
print("File successfully saved as 'engineered_iot_weather_dataset.csv'!")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# 1. Define original sensor features & target variable
# Using only the original features as requested for the baseline
base_features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
target = 'Machine failure'  
# Assuming 'Machine failure' stands for Machine Failure 
# Filter features that exist in df to avoid any KeyErrors
base_features = [col for col in base_features if col in df.columns]

In [ ]:
# 2. Prepare the dataset
# Drop any rows where features or target might have missing values
baseline_df = df[base_features + [target]].dropna()

X = baseline_df[base_features]
y = baseline_df[target]

In [ ]:
# 3. Train-Test Split (Stratified to handle class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
# 4. Initialize SMOTE and resample the training data
print("Applying SMOTE and Training Random Forest...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Train using the resampled data
baseline_rf = RandomForestClassifier(random_state=42, n_estimators=100)
baseline_rf.fit(X_train_resampled, y_train_resampled)

# 5. Make Predictions (Keep testing on original X_test)
y_pred = baseline_rf.predict(X_test)


# 6. Evaluate and print scores
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='binary') # Use 'macro' if it is multi-class

print("\n================ BASELINE MODEL METRICS ================")
print(f"Accuracy Score : {accuracy:.4f}")
print(f"F1-Score       : {f1:.4f}")
print("========================================================")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# 1. Apply SMOTE and Train
print("Applying SMOTE + Probability Tuning...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

baseline_rf = RandomForestClassifier(random_state=42, n_estimators=100)
baseline_rf.fit(X_train_resampled, y_train_resampled)

# 2. Use predict_proba instead of predict
y_probs = baseline_rf.predict_proba(X_test)[:, 1]

# 3. Tweak this value (Try 0.6 or 0.7 to raise precision)
custom_threshold = 0.65  
y_pred = (y_probs >= custom_threshold).astype(int)

# Evaluate and print scores
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='binary')

print("\n================ NEW TUNED MODEL METRICS ================")
print(f"Accuracy Score : {accuracy:.4f}")
print(f"F1-Score       : {f1:.4f}")
print("========================================================")
print("\nClassification Report:\n", classification_report(y_test, y_pred))



In [ ]:
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score

In [ ]:
# 1. Import Advanced Boosting Algorithms
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    'XGBoost': XGBClassifier(),
    'LightGBM': LGBMClassifier(),
    'CatBoost': CatBoostClassifier()
}

print("Initializing Advanced Model Exploration Framework...")

# 2. Define the Pipelines (SMOTE + Model)
# Idhu tuning pannum podhu data leak aagama automation-ah SMOTE panni train panni paakum.
pipelines = {
    'XGBoost': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0))
    ]),
    'LightGBM': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', LGBMClassifier(random_state=42, verbose=-1))
    ]),
    'CatBoost': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', CatBoostClassifier(random_state=42, verbose=0))
    ])
}

In [ ]:
# 3. Define Hyperparameter Tuning Grids
# Hyperparameters-ah limit panni set panyiruken, appo dhaan GridSearchCV seekiram run aagum.
param_grids = {
    'XGBoost': {
        'model__n_estimators': [0.05, 0.1],
        'model__max_depth': [0.05, 0.1],
        'model__learning_rate': [0.05, 0.1]
    },
    'LightGBM': {
        'model__n_estimators': [0.05, 0.1],
        'model__max_depth': [0.05, 0.1],
        'model__learning_rate': [0.05, 0.1]
    },
    'CatBoost': {
        'model__iterations': [0.05, 0.1],
        'model__depth': [0.05, 0.1],
        'model__learning_rate': [0.05, 0.1]
    }
}


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report

# 4. Dictionary to store best models
best_models = {}

# 1. Pipelines definition
# Handle imbalanced data by setting class_weight='balanced' to give more weight to minority class failures
pipelines = {
    'lr': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ]),
    'rf': Pipeline([
        ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
    ])
}

# 2. Param Grids definition
# Specified appropriate estimator counts for RandomForest to avoid non-finite or invalid grid search fits
param_grids = {
    'lr': {
        'classifier__C': [0.01, 0.1, 1, 10]
    },
    'rf': {
        'classifier__n_estimators': [None, 10, 20],  
        'classifier__max_depth': [None, 10, 20]
    }
}

# 3. Run GridSearchCV for each model
for name in pipelines.keys():
    print(f"\nTuning hyperparameters for {name} using GridSearchCV...")
    
    grid_search = GridSearchCV(
        estimator=pipelines[name],
        param_grid=param_grids[name],
        cv=3,                 # 3-Fold Cross Validation
        scoring='f1',         # Optimize based on F1-Score
        n_jobs=-1,            # Use all available CPU cores for speed
        verbose=1
    )
    # Train the Grid Search
    grid_search.fit(X_train, y_train)
    
    # Save the best estimator
    best_models[name] = grid_search.best_estimator_
    print(f"Best Params for {name}: {grid_search.best_params_}")

# 4. Evaluate All Tuned Advanced Models on Test Data
print("\n================ ADVANCED MODELS EVALUATION ================")
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='binary')
    
    print(f"\n--- {name} Performance ---")
    print(f"Optimized F1-Score: {f1:.4f}")
    print(classification_report(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(8, 6))

for name, model in best_models.items():
    # Model probabilities (Class 1 probabilities only)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        # Decision function for models without predict_proba
        y_prob = model.decision_function(X_test)
        
    # Compute False Positive Rate and True Positive Rate
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    
    # Plot the ROC curve for each model
    plt.plot(fpr, tpr, label=f'{name.upper()} (AUC = {auc_score:.4f})', linewidth=2)

# Reference diagonal line (Random guessing line)
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing (AUC = 0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.title('ROC Curve - Model Comparison')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
